# Transformación de datos del Censo Nacional de Población y Vivienda 2018

Este notebook utiliza PySpark para transformar los datos del Censo Nacional de Población y Vivienda (CNPV) 2018 del DANE. La información se organiza por municipio, aplicando reglas de calidad y validación para preparar los datos para su posterior análisis. Los registros originales se conservan sin modificaciones.
.


## 1. Configuración

Se define parámetros necesarios para ejecutar la transformación. La escritura en PostgreSQL permanece desactivada mientras se validan los resultados.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

NB_DIR = Path("/home/jovyan/work/spark-apps/notebooks")
if not NB_DIR.exists():
    NB_DIR = Path.cwd().resolve()
    if NB_DIR.name == "transformation":
        NB_DIR = NB_DIR.parent
SRC = NB_DIR.parent / "src"
sys.path.insert(0, str(SRC))

from transformation.spark_sesion import _asegurar_pyspark, crear_spark

_asegurar_pyspark()
from pyspark.sql import functions as F
from transformation.controles import CONTROLES_CRITICOS_DANE, controles_criticos_ok
from transformation.dane import ANIO_CENSO, invariantes_poblacion
from transformation.io import persistir_poblacion, verificar_persistencia
from transformation.jobs.job_dane import ejecutar

ANIO = ANIO_CENSO
ESCRIBIR = False  # True solo tras validar conteos

SALIDAS = NB_DIR / "salidas" / "transformation" / str(ANIO) / "dane"
FIGURAS = SALIDAS / "figuras"
SALIDAS.mkdir(parents=True, exist_ok=True)
FIGURAS.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.titlesize"] = 11

print("Año censal:", ANIO)
print("ESCRIBIR:", ESCRIBIR)
print("Salidas:", SALIDAS)
print("CNPV_PARQUET:", os.environ.get("CNPV_PARQUET", "/datalake/cnpv2018/personas"))
print("PGUSER definido:", bool(os.environ.get("PGUSER")))


## 2. Spark local

Se inicia Spark

In [ ]:
spark = crear_spark("nb08-transformacion-dane")
print("Spark:", spark.version, spark.sparkContext.master)
res = ejecutar(spark, anio=ANIO, escribir=False)
pob = res["poblacion"]
cuadro = pd.DataFrame(res["cuadro"])
print("Municipios derivados:", pob.count())
print("Invariantes:", res["fallos_invariante"] or "ok")
display(cuadro)
cuadro.to_csv(SALIDAS / "control_reconciliacion.csv", index=False)


## 3. Homologación territorial con DIVIPOLA

Se comparan los códigos municipales del censo con la clasificación DIVIPOLA del DANE. Se identifican los municipios que tienen correspondencia y aquellos que presentan diferencias o no cuentan con información asociada.

In [ ]:
hom = (pob.groupBy("estado_homologacion", "en_divipola")
       .agg(F.count("*").alias("municipios"), F.sum("poblacion").alias("personas"))
       .orderBy(F.col("personas").desc()).toPandas())
display(hom)
hom.to_csv(SALIDAS / "homologacion.csv", index=False)

noh = res["no_homologados"].orderBy(F.col("poblacion").desc()).toPandas()
print("Códigos sin DIVIPOLA:", len(noh))
display(noh)
noh.to_csv(SALIDAS / "codigos_no_homologados.csv", index=False)

faltan = res["divipola_sin_censo"].toPandas()
print("DIVIPOLA sin personas 2018:", len(faltan))
display(faltan)
faltan.to_csv(SALIDAS / "divipola_sin_censo.csv", index=False)


## 4. Validación de sexo, edad y zona

Se revisan las categorías de sexo, edad y zona de residencia para verificar que correspondan con las definiciones del DANE. También se calculan los totales de población según estas características.

In [ ]:
cats = res["categorias"]
print("Columnas:", cats["sexo_columna"], cats["edad_columna"], cats["zona_columna"])
print("sexo ok:", cats["categorias_sexo_ok"], cats["sexo"])
print("edad ok:", cats["categorias_edad_ok"], "n_cats", len(cats["edad"]))
print("zona ok:", cats["categorias_zona_ok"], cats["zona"])
pd.DataFrame([
    {"campo": "sexo", "columna": cats["sexo_columna"], "ok": cats["categorias_sexo_ok"], "valores": str(cats["sexo"])},
    {"campo": "edad", "columna": cats["edad_columna"], "ok": cats["categorias_edad_ok"], "valores": str(sorted(cats["edad"]))},
    {"campo": "zona", "columna": cats["zona_columna"], "ok": cats["categorias_zona_ok"], "valores": str(cats["zona"])},
]).to_csv(SALIDAS / "categorias.csv", index=False)

tot = pob.agg(
    F.sum("poblacion").alias("poblacion"),
    F.sum("pob_hombres").alias("hombres"),
    F.sum("pob_mujeres").alias("mujeres"),
    F.sum("pob_0_14").alias("e0_14"),
    F.sum("pob_15_64").alias("e15_64"),
    F.sum("pob_65_mas").alias("e65"),
    F.sum("pob_cabecera").alias("cabecera"),
    F.sum("pob_centro_poblado").alias("centro"),
    F.sum("pob_rural_disperso").alias("rural"),
    F.sum("pob_resto").alias("resto"),
).toPandas()
display(tot)
tot.to_csv(SALIDAS / "totales_indicadores.csv", index=False)


## 5. Gráficos de validación

Se presentan gráficos para visualizar los municipios con mayor población, los resultados de la homologación territorial y la distribución de la población por sexo, cuando los datos permiten realizar esta comparación.

In [ ]:
top = (pob.filter(F.col("en_divipola"))
       .select("cod_municipio", "nombre_municipio_divipola", "poblacion")
       .orderBy(F.col("poblacion").desc()).limit(12).toPandas())
top.to_csv(SALIDAS / "top_municipios.csv", index=False)
fig, ax = plt.subplots()
ax.barh(top["nombre_municipio_divipola"], top["poblacion"] / 1e6, color="#4C78A8")
ax.invert_yaxis()
ax.set_xlabel("Millones de personas (2018)")
ax.set_title("Población municipal CNPV 2018 (top 12)")
plt.tight_layout()
plt.savefig(FIGURAS / "top_municipios.png", dpi=120, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots()
ax.bar(hom["estado_homologacion"], hom["personas"] / 1e6, color="#72B7B2")
ax.set_ylabel("Millones de personas")
ax.set_title("Personas 2018 según cruce DIVIPOLA")
plt.tight_layout()
plt.savefig(FIGURAS / "homologacion_personas.png", dpi=120, bbox_inches="tight")
plt.show()

if cats["categorias_sexo_ok"] and tot["hombres"].notna().all():
    fig, ax = plt.subplots()
    ax.bar(["Hombres", "Mujeres"], [tot.loc[0, "hombres"] / 1e6, tot.loc[0, "mujeres"] / 1e6], color=["#4C78A8", "#F58518"])
    ax.set_ylabel("Millones")
    ax.set_title("Sexo (CNPV 2018)")
    plt.tight_layout()
    plt.savefig(FIGURAS / "sexo.png", dpi=120, bbox_inches="tight")
    plt.show()


## 6. Persistencia (preparada, no se ejecuta)

Se define el procedimiento para almacenar la población municipal transformada en PostgreSQL, incluyendo las validaciones necesarias antes de publicar la información

In [ ]:
print("ESCRIBIR =", ESCRIBIR)
print("tfm.poblacion_municipal:", res["observado"]["municipios"], "filas")
print("Suma personas:", res["observado"]["personas_agregadas"])
if ESCRIBIR:
    if not controles_criticos_ok(res["cuadro"], CONTROLES_CRITICOS_DANE):
        print("Control crítico distinto del EDA. No se publica.")
    elif res.get("fallos_invariante"):
        print("Invariantes fallidas. No se publica:", res["fallos_invariante"])
    else:
        persistir_poblacion(pob, res["control"], ANIO, res["observado"]["personas_parquet"])
        verif = verificar_persistencia(
            spark, str(ANIO), {"poblacion_municipal": res["observado"]["municipios"]},
            pks={"poblacion_municipal": ["anio", "cod_municipio"]},
            col_tiempo="anio",
        )
        display(pd.DataFrame(verif))
        print("Escritura tfm.poblacion_municipal hecha.")
else:
    print("Sin escritura. Para persistir: ESCRIBIR=true en job_dane.py")
pd.DataFrame([{
    "tabla": "tfm.poblacion_municipal",
    "filas": res["observado"]["municipios"],
    "grano": "anio + cod_municipio",
    "anio": ANIO,
}]).to_csv(SALIDAS / "tablas_pendientes_tfm.csv", index=False)


## 7. Cierre


In [ ]:
print("Parquet CNPV: solo lectura.")
print("Mongo cnpv2018_personas / fact_poblacion: no usados.")
print("tfm.proceso / tfm.contrato / public: no se tocan.")
print("Anomalías y asistente: no implementados aquí.")
spark.stop()
print("SparkSession cerrada.")
